In [57]:
import pandas as pd
import numpy as np

#Preprocessing
from sklearn.preprocessing import RobustScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from scipy.stats import uniform

#Model Machine Learning
from sklearn.linear_model import OrthogonalMatchingPursuit
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,mean_absolute_percentage_error,r2_score
import warnings
warnings.filterwarnings('ignore',category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Orthogonal matching pursuit*")

## 1.Orthogonal Matching Pursuit Regression (OMP)

In [58]:
df = pd.read_csv('../0.dataset/dataset_California_HousePrice_clean.csv')
df_x = df.drop(columns='median_house_value')
df_y = df['median_house_value']

In [59]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [60]:
feature_numerik = X_train.select_dtypes(include=np.number).columns
feature_categori = X_train.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('numerik_scaler',RobustScaler(),feature_numerik),
        ('categori_encoding',OneHotEncoder(drop='first', handle_unknown='ignore'),feature_categori)
    ]
)

In [61]:
selector_model = DecisionTreeRegressor(max_depth=2,random_state=42)
model_ridgeRegression = Pipeline([
    ('preprocessing',preprocessor),
    ('feature_selection',SequentialFeatureSelector(estimator=selector_model,direction='forward')),
    ('model_regression',OrthogonalMatchingPursuit())
])

In [62]:
params={
    'feature_selection__n_features_to_select': [5], # hindari 'auto' jika memicu terlalu banyak fitur
    
    # Menyesuaikan n_iter agar lebih konservatif
    'model_regression__n_nonzero_coefs': [None, 1, 2, 3, 4, 5], 
    'model_regression__tol': [None, 1e-3, 1e-2], 
    'model_regression__fit_intercept': [True, False]
}
random_search = RandomizedSearchCV(estimator=model_ridgeRegression,param_distributions=params,n_iter=5,cv=5,scoring='r2',n_jobs=-1,verbose=1)
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


,estimator,Pipeline(step...ngPursuit())])
,param_distributions,"{'feature_selection__n_features_to_select': [5], 'model_regression__fit_intercept': [True, False], 'model_regression__n_nonzero_coefs': [None, 1, ...], 'model_regression__tol': [None, 0.001, ...]}"
,n_iter,5
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [63]:
best_model_pipeline = random_search.best_estimator_

preprocessor_step = best_model_pipeline.named_steps['preprocessing']
sfs_step = best_model_pipeline.named_steps['feature_selection']

kolom_setelah_transformasi = preprocessor_step.get_feature_names_out()
fitur_terpilih = kolom_setelah_transformasi[sfs_step.get_support()]

print(f'\nHyperparameter Terbaik:\n{random_search.best_params_}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

#cek score
r2_score_train = best_model_pipeline.score(X_train, y_train)
r2_score_test = best_model_pipeline.score(X_test, y_test)

y_pred_test = best_model_pipeline.predict(X_test)
y_pred_train = best_model_pipeline.predict(X_train)

print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')



Hyperparameter Terbaik:
{'model_regression__tol': 0.001, 'model_regression__n_nonzero_coefs': 5, 'model_regression__fit_intercept': True, 'feature_selection__n_features_to_select': 5}

Fitur Terbaik Yang Terpilih:
['numerik_scaler__longitude', 'numerik_scaler__latitude', 'numerik_scaler__housing_median_age', 'numerik_scaler__median_income', 'categori_encoding__ocean_proximity_INLAND']

Akurasi Validasi Terbaik: 60.93%


In [64]:
def evaluate_model(pred,test,score_umum,evaluate_type,model_name='Simple LinearRegression'):
    r2 = r2_score(pred,test)
    mse = mean_squared_error(pred,test)
    mae = mean_absolute_error(pred,test)
    rmse = root_mean_squared_error(pred,test)
    mape = mean_absolute_percentage_error(pred,test)

    data = {
        'Evaluated Data': [evaluate_type],
        'Score Umum': [score_umum*100],
        'R2 Score': [r2],
        'MAE': [mae],
        'MSE': [mse],
        'RMSE': [rmse],
        'MAPE': [mape]
    }

    df_result = pd.DataFrame(data,index=[model_name])
    return df_result

In [65]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    r2_train_score = df_eval_train['R2 Score'].values[0]
    r2_test_score = df_eval_test['R2 Score'].values[0]
    gap = r2_train_score -  r2_test_score

    if  r2_score_train < 0.60 or r2_test_score < 0.60:
        status = 'Underfitting (Performa Rendah pada Kedua Data)'
    elif gap > 0.05:
        status = f'Overfitting (Gap Terlalu Jauh: {gap*100:.2f}%)'
    elif gap < -0.05:
        status = f'Data Leakage / Bias (Test > Train, Gap: {gap*100:.2f}%)'
    else:
        status = 'Good Fit (Model Stabil)'

    df_combined['Status'] = status
    return df_combined

In [66]:
df_eval_train = evaluate_model(y_pred_train,y_train,score_umum=r2_score_train,evaluate_type='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,score_umum=r2_score_test,evaluate_type='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print(f"{'=' * 50} PREDIKSI DENGAN SAMPLE DATASET {'=' * 68}")
print(df_eval.to_string())
print("\n" + "="* 150 + "\n")

================================================== PREDIKSI DENGAN SAMPLE DATASET ====================================================================
  Evaluated Data  Score Umum  R2 Score           MAE           MSE          RMSE      MAPE                                          Status
0       Training   60.971358  0.359886  53034.093883  5.217264e+09  72230.629123  0.315933  Underfitting (Performa Rendah pada Kedua Data)
1           Test   59.282572  0.344817  52996.217034  5.335648e+09  73045.522048  0.314642  Underfitting (Performa Rendah pada Kedua Data)




In [67]:
data = {
    "longitude": [-122.23, -122.22, -122.24],
    "latitude": [37.88, 37.86, 37.85],
    "housing_median_age": [41, 21, 52],
    "total_rooms": [880, 7099, 1467],
    "total_bedrooms": [129.0, 1106.0, 190.0],
    "population": [322, 2401, 496],
    "households": [126, 1138, 177],
    "median_income": [8.3252, 8.3014, 7.2574],
    "ocean_proximity": ["NEAR BAY", "NEAR BAY", "NEAR BAY"],
    "median_house_value": [452600, 358500, 352100],
}
df_new = pd.DataFrame(data)
data_rumahBaru_x = df_new.drop(columns='median_house_value')
data_rumahBaru_y = df_new['median_house_value']

In [68]:
print("=== Melakukan Prediksi Data Harga Rumah Baru ===")
prediksi_rumah = best_model_pipeline.predict(data_rumahBaru_x)

hasil_df = data_rumahBaru_x.copy()
hasil_df['Harga Prediksi'] = prediksi_rumah
hasil_df['Harga Asli'] = data_rumahBaru_y
hasil_df['Selisih Harga Asli VS Prediksi'] = hasil_df['Harga Prediksi'] - hasil_df['Harga Asli']

akurasi_bawaan = best_model_pipeline.score(data_rumahBaru_x,data_rumahBaru_y)
df_eval_data_baru = evaluate_model(
    pred=prediksi_rumah,
    test=data_rumahBaru_y,
    score_umum=akurasi_bawaan,
    evaluate_type='Data_Baru'
)
print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))
hasil_df[['Harga Prediksi','Harga Asli','Selisih Harga Asli VS Prediksi']]

=== Melakukan Prediksi Data Harga Rumah Baru ===
Akurasi Model: 38.18%

Tabel Skor Evaluasi Lengkap Data Baru:
Evaluated Data  Score Umum  R2 Score          MAE          MSE         RMSE     MAPE
     Data_Baru   38.181232 -7.759202 35951.615129 1.304790e+09 36121.870284 0.090314


,Harga Prediksi,Harga Asli,Selisih Harga Asli VS Prediksi
0,413051.100232,452600,-39548.899768
1,395602.715349,358500,37102.715349
2,383303.230271,352100,31203.230271
